# 01 — Feature Engineering Pipeline: Encoding, Scaling, Imputation, Selection

**Overview:** a mixed numerical+categorical synthetic dataset (with missing values and an
outlier) fully preprocessed via `ColumnTransformer` + `Pipeline`, including feature selection,
cross-validated leakage-safely.

**Required libraries:** `numpy`, `pandas`, `scikit-learn`.

Connects to: `12_ml_evaluation` (`Pipeline`-based leakage-safe cross-validation). Closes Day 2's
"classic ML" sequence before `14_hyperparameter_tuning`.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **ML** | Machine Learning | Learning patterns from data to make predictions, instead of hand-coding rules. |
| **CV** | Cross-Validation | Repeatedly splitting data into train/validation folds to estimate generalisation. |

## 0. Environment & a deliberately messy mixed dataset

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

rng = np.random.default_rng(42)
n = 300

df = pd.DataFrame({
    "age": rng.normal(40, 12, size=n).round(1),
    "income": rng.lognormal(mean=10.5, sigma=0.4, size=n).round(0),
    "education": rng.choice(["high_school", "bachelors", "masters", "phd"], size=n,
                             p=[0.4, 0.35, 0.2, 0.05]),   # ordinal: genuine order
    "city": rng.choice([f"city_{i}" for i in range(8)], size=n),  # nominal: no order
})

# True label depends on age, income, and education level (not city -- a deliberately
# uninformative feature, useful for the feature-selection section).
education_rank = df["education"].map({"high_school": 0, "bachelors": 1, "masters": 2, "phd": 3})
score = 0.03 * df["age"] + 0.00002 * df["income"] + 0.8 * education_rank
label = (score + rng.normal(scale=1.0, size=n) > score.median()).astype(int)
df["label"] = label

# Inject missing values and one outlier, on purpose.
missing_idx = rng.choice(n, size=20, replace=False)
df.loc[missing_idx, "income"] = np.nan
df.loc[0, "age"] = 250.0   # an obviously bad outlier (data-entry error)

print(df.head())
print("\nmissing values:\n", df.isna().sum())
print("\nclass balance:", df["label"].value_counts(normalize=True).to_dict())


     age   income    education    city  label
0  250.0  72471.0    bachelors  city_0      1
1   27.5  19662.0      masters  city_2      1
2   49.0  51304.0  high_school  city_3      1
3   51.3  31844.0    bachelors  city_1      1
4   16.6  35436.0      masters  city_0      0

missing values:
 age           0
income       20
education     0
city          0
label         0
dtype: int64

class balance: {1: 0.5266666666666666, 0: 0.47333333333333333}


## 1. Why this needs different handling per column

💡 **Key Idea:** `education` is ORDINAL (high_school < bachelors < masters < phd — a real
order worth preserving as a single integer); `city` is NOMINAL (no order — one-hot encode it
instead, or an ordinal encoder would invent a false ranking).

In [2]:
numerical_features = ["age", "income"]
ordinal_features = ["education"]
nominal_features = ["city"]

education_order = [["high_school", "bachelors", "masters", "phd"]]
print("Treating 'education' as ordinal with explicit order:", education_order[0])
print("Treating 'city' as nominal (one-hot) -- no implied order among city_0..city_7.")


Treating 'education' as ordinal with explicit order: ['high_school', 'bachelors', 'masters', 'phd']
Treating 'city' as nominal (one-hot) -- no implied order among city_0..city_7.


## 2. Handling the outlier (deliberately, not by default)

⚠️ **Common Mistake:** silently feeding an obvious data-entry error (age=250) into a scaler,
which will badly distort the mean/std used to scale every other value.

In [3]:
print("age column stats BEFORE capping the outlier:")
print(df["age"].describe())

# Cap (winsorize) ages outside a plausible human range, rather than deleting the whole row.
df["age"] = df["age"].clip(lower=0, upper=100)

print("\nage column stats AFTER capping:")
print(df["age"].describe())
assert df["age"].max() <= 100


age column stats BEFORE capping the outlier:
count    300.000000
mean      40.195667
std       16.500642
min        9.200000
25%       31.950000
50%       38.950000
75%       45.800000
max      250.000000
Name: age, dtype: float64

age column stats AFTER capping:
count    300.000000
mean      39.695667
std       11.694664
min        9.200000
25%       31.950000
50%       38.950000
75%       45.800000
max      100.000000
Name: age, dtype: float64


## 3. Build the ColumnTransformer + Pipeline

🎯 **Interview Tip:** each branch of the `ColumnTransformer` gets its own imputer/encoder/
scaler, and the whole thing refits per cross-validation fold automatically when wrapped in a
`Pipeline` -- exactly the leakage-safe pattern from `12_ml_evaluation`.

In [4]:
numerical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

ordinal_pipeline = Pipeline([
    ("encode", OrdinalEncoder(categories=education_order)),
])

nominal_pipeline = Pipeline([
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("ordinal", ordinal_pipeline, ordinal_features),
    ("nominal", nominal_pipeline, nominal_features),
])

full_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("select", SelectKBest(score_func=f_classif, k=5)),
    ("model", LogisticRegression(max_iter=1000)),
])

X = df.drop(columns=["label"])
y = df["label"]

full_pipeline.fit(X, y)
print("pipeline fit successfully.")
print("transformed feature count after preprocessing (before selection):",
      preprocessor.fit_transform(X).shape[1])


pipeline fit successfully.
transformed feature count after preprocessing (before selection): 11


## 4. Cross-validate the whole pipeline (leakage-safe by construction)

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(full_pipeline, X, y, cv=cv, scoring="accuracy")

print("5-fold CV accuracy:", cv_scores.round(3))
print(f"mean={cv_scores.mean():.3f}, std={cv_scores.std():.3f}")

assert cv_scores.mean() > 0.55   # should beat random guessing meaningfully


5-fold CV accuracy: [0.833 0.733 0.683 0.617 0.667]
mean=0.707, std=0.073


## 5. What did feature selection keep?

💡 **Intuition:** `city` was constructed to be uninformative -- a well-functioning
`SelectKBest` should mostly deprioritize the one-hot `city_*` columns relative to age/income/education.

In [6]:
preprocessor_fitted = full_pipeline.named_steps["preprocess"]
selector_fitted = full_pipeline.named_steps["select"]

feature_names = (
    numerical_features
    + ordinal_features
    + list(preprocessor_fitted.named_transformers_["nominal"]
           .named_steps["encode"].get_feature_names_out(nominal_features))
)
selected_mask = selector_fitted.get_support()
selected_features = np.array(feature_names)[selected_mask]

print("all feature names after preprocessing:", feature_names)
print("\nfeatures kept by SelectKBest(k=5):", list(selected_features))
print("\nage/income/education should dominate the kept features, since they actually drive the label.")

assert "age" in selected_features or "income" in selected_features


all feature names after preprocessing: ['age', 'income', 'education', 'city_city_0', 'city_city_1', 'city_city_2', 'city_city_3', 'city_city_4', 'city_city_5', 'city_city_6', 'city_city_7']

features kept by SelectKBest(k=5): [np.str_('age'), np.str_('income'), np.str_('education'), np.str_('city_city_4'), np.str_('city_city_7')]

age/income/education should dominate the kept features, since they actually drive the label.


## 6. Connection to what's next

This same `Pipeline` + cross-validation setup is exactly what `14_hyperparameter_tuning` wraps
a grid/random search around next — tuning `SelectKBest`'s `k`, the scaler choice, or the
model's regularization strength, all without ever leaking fold information.

## 7. Interview questions
1. Why does `education` get an ordinal encoding while `city` gets one-hot encoding?
2. What would go wrong if you fit the `SimpleImputer` and `StandardScaler` on the full dataset before cross-validation instead of inside the `Pipeline`?
3. Why cap (winsorize) the outlier instead of deleting that row entirely?
4. What does `SelectKBest(k=5)` actually select, and based on what criterion here (`f_classif`)?
5. Why does wrapping everything in one `Pipeline` object matter for correctness, not just convenience?

## 8. Exercises
1. Add a target-encoded version of `city` (mean label rate per city) and compare cross-validated performance to the one-hot version, making sure the target encoding is fit per-fold (hint: a custom transformer or `category_encoders.TargetEncoder` inside the pipeline).
2. Change `SelectKBest`'s `k` from 3 to 8 and plot cross-validated accuracy vs `k`.
3. Replace `StandardScaler` with `MinMaxScaler` for the numerical features and compare cross-validated performance.

## 9. Summary

✅ Numerical and categorical features need different preprocessing; `ColumnTransformer` applies
the right preprocessing per column type; wrapping everything (preprocessing + selection + model)
in a `Pipeline` makes cross-validation automatically leakage-safe. This closes
`13_feature_engineering` — next: `14_hyperparameter_tuning`.